# Agentic ML Workflows - Building AI Agents with TuiML

This tutorial shows how to build **agentic ML workflows** where an LLM uses TuiML tools to autonomously discover algorithms, run experiments, train models, and deploy them.

## What You'll Learn

1. **Why Agentic ML** - LLM + TuiML tools = automated ML pipelines
2. **Exploring Available Tools** - Discovery and workflow tool categories
3. **Tool Schemas** - Understanding JSON Schema for each tool
4. **Executing Tools Programmatically** - Calling `tool["function"]` directly
5. **Simulated Agent Workflow** - End-to-end agent finding the best classifier
6. **Config-Driven Workflows** - Using `tuiml.run()` with LLM-generated configs
7. **API Schema Discovery** - `get_api_info()` for programmatic introspection
8. **Integration Patterns** - Anthropic Claude and OpenAI function calling formats
9. **MCP Integration** - Connecting to Claude Desktop via MCP

---

## 1. Why Agentic ML?

Traditional ML workflows require a human to manually:
- Choose algorithms
- Set hyperparameters
- Run experiments
- Interpret results

**Agentic ML** flips this: an LLM agent receives a high-level goal (e.g., "find the best classifier for this dataset") and autonomously uses **tools** to accomplish it.

TuiML provides 12 purpose-built tools that an LLM can call:

| Category | Tools | Purpose |
|----------|-------|---------|
| **Discovery** | `tuiml_list`, `tuiml_describe`, `tuiml_search` | Find and understand components |
| **Workflow** | `tuiml_train`, `tuiml_predict`, `tuiml_evaluate` | Train, predict, evaluate |
| **Experiment** | `tuiml_experiment` | Compare multiple algorithms |
| **Data** | `tuiml_upload_data` | Upload inline datasets |
| **Persistence** | `tuiml_save_model` | Save models to disk |
| **Serving** | `tuiml_serve_model`, `tuiml_stop_server`, `tuiml_server_status` | Deploy models as REST APIs |

Each tool is a Python dict with `name`, `description`, `input_schema`, and a callable `function` key. The agent calls the function, gets structured JSON back, reasons about the result, and decides what to do next.

---

## 2. Exploring Available Tools

TuiML organizes its LLM tools into two groups:
- **DISCOVERY_TOOLS** - for finding and understanding components
- **WORKFLOW_TOOLS** - for training, predicting, evaluating, and serving models

In [1]:
from tuiml.llm.tools import DISCOVERY_TOOLS, WORKFLOW_TOOLS

all_tools = {**DISCOVERY_TOOLS, **WORKFLOW_TOOLS}

print(f"Total LLM tools: {len(all_tools)}")
print()

print("=== Discovery Tools ===")
for name, schema in DISCOVERY_TOOLS.items():
    print(f"  {name}")
    print(f"    {schema['description'][:80]}...")
    print()

print("=== Workflow Tools ===")
for name, schema in WORKFLOW_TOOLS.items():
    print(f"  {name}")
    print(f"    {schema['description'][:80]}...")
    print()

Total LLM tools: 12

=== Discovery Tools ===
  tuiml_list
    List all available TuiML components (algorithms, preprocessors, datasets, fea...

  tuiml_describe
    Get detailed information and parameter schema for any TuiML component....

  tuiml_search
    Search for components by keyword in name or description....

=== Workflow Tools ===
  tuiml_train
    Train a machine learning model. Supports supervised (classifiers, regressors) an...

  tuiml_predict
    Make predictions using a trained model on new data....

  tuiml_evaluate
    Evaluate a trained model on test data and compute metrics....

  tuiml_experiment
    Compare multiple algorithms on a dataset with cross-validation and statistical t...

  tuiml_upload_data
    Upload dataset content directly (CSV or ARFF text). Returns a file path that can...

  tuiml_save_model
    Copy a trained model to a custom path. Use this when the user wants to save or d...

  tuiml_serve_model
    Start a REST API server to serve a trained mo

---

## 3. Tool Schemas

Each tool carries a JSON Schema that describes its inputs. This is what an LLM reads to decide how to call the tool.

In [2]:
import json

# Inspect the tuiml_train tool schema
train_schema = WORKFLOW_TOOLS["tuiml_train"]

print("tuiml_train")
print(f"Description: {train_schema['description']}")
print()
print("Input Schema:")
print(json.dumps(train_schema["inputSchema"], indent=2))

tuiml_train
Description: Train a machine learning model. Supports supervised (classifiers, regressors) and unsupervised (clusterers) learning. Returns trained model and metrics.

Input Schema:
{
  "type": "object",
  "properties": {
    "algorithm": {
      "type": "string",
      "description": "Algorithm class name. Examples:\n- Classifiers: 'RandomForestClassifier', 'SVM', 'NaiveBayesClassifier', 'C45TreeClassifier'\n- Regressors: 'LinearRegression', 'M5ModelTreeRegressor'\n- Clusterers: 'KMeansClusterer', 'GaussianMixtureClusterer', 'DBSCANClusterer'"
    },
    "data": {
      "type": "string",
      "description": "Data file path or built-in dataset name (e.g., 'iris', 'wine')"
    },
    "target": {
      "type": "string",
      "description": "Target column (required for supervised, optional for clustering)"
    },
    "preprocessing": {
      "type": "array",
      "items": {
        "oneOf": [
          {
            "type": "string"
          },
          {
            "type

In [3]:
# Inspect the tuiml_experiment tool schema
exp_schema = WORKFLOW_TOOLS["tuiml_experiment"]

print("tuiml_experiment")
print(f"Description: {exp_schema['description']}")
print()
print("Input Schema:")
print(json.dumps(exp_schema["inputSchema"], indent=2))

tuiml_experiment
Description: Compare multiple algorithms on a dataset with cross-validation and statistical tests. Supports supervised learning (classification, regression) and unsupervised learning (clustering).

Input Schema:
{
  "type": "object",
  "properties": {
    "algorithms": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "List of algorithm class names to compare (e.g., ['RandomForestClassifier', 'SVM'] for classification, ['KMeansClusterer', 'GaussianMixtureClusterer'] for clustering)"
    },
    "data": {
      "type": "string",
      "description": "Path to dataset file or built-in dataset name (e.g., 'iris', 'wine')"
    },
    "target": {
      "type": "string",
      "description": "Target column name (for supervised learning)"
    },
    "cv": {
      "type": "integer",
      "default": 10,
      "description": "Number of CV folds (ignored for clustering)"
    },
    "metrics": {
      "type": "array",
      "items": {


In [4]:
# Inspect a discovery tool: tuiml_describe
describe_schema = DISCOVERY_TOOLS["tuiml_describe"]

print("tuiml_describe")
print(f"Description: {describe_schema['description']}")
print()
print("Input Schema:")
print(json.dumps(describe_schema["inputSchema"], indent=2))

tuiml_describe
Description: Get detailed information and parameter schema for any TuiML component.

Input Schema:
{
  "type": "object",
  "properties": {
    "name": {
      "type": "string",
      "description": "Component name (e.g., 'RandomForestClassifier', 'SimpleImputer', 'iris')"
    }
  },
  "required": [
    "name"
  ]
}


---

## 4. Executing Tools Programmatically

TuiML provides `execute_tool()` to call any tool by name. This is the same function that the MCP server and LLM integrations use internally.

In [5]:
from tuiml.llm.tools import execute_tool

# Call tuiml_list to see available classifiers
result = execute_tool(
    "tuiml_list",
    category="algorithm",
    type="classifier",
    limit=8
)

print(f"Status: {result['status']}")
print(f"Total classifiers: {result['total']}")
print(f"Showing first {result['count']}:")
for comp in result["components"]:
    print(f"  - {comp['name']}: {comp['description'][:60]}...")

Status: success
Total classifiers: 70
Showing first 8:
  - tuiml_algorithm_NaiveBayesClassifier: Naive Bayes classifier using **pluggable probability estimat...
  - tuiml_algorithm_NaiveBayesMultinomialClassifier: Multinomial Naive Bayes classifier for **text** and **discre...
  - tuiml_algorithm_BayesianNetworkClassifier: Bayesian Network classifier using **probabilistic graphical ...
  - tuiml_algorithm_DecisionStumpClassifier: Decision Stump - a one-level decision tree....
  - tuiml_algorithm_C45TreeClassifier: C4.5 Decision Tree classifier....
  - tuiml_algorithm_RandomTreeClassifier: Random Tree classifier - a single randomized decision tree....
  - tuiml_algorithm_RandomForestClassifier: Random Forest classifier - ensemble of random trees....
  - tuiml_algorithm_ReducedErrorPruningTreeClassifier: Reduced Error Pruning Tree classifier....


In [6]:
# Call tuiml_train to train a model
result = execute_tool(
    "tuiml_train",
    algorithm="RandomForestClassifier",
    data="iris",
    target="class",
    cv=5
)

print(f"Status: {result['status']}")
print(f"Model ID: {result.get('model_id')}")
print(f"Model class: {result.get('model_class')}")
print(f"Metrics: {result.get('metrics')}")

# Save the model_id for later
rf_model_id = result.get("model_id")

Status: success
Model ID: a43676fc3139
Model class: RandomForestClassifier
Metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9395583160800551, 'cv_f1_score_std': 0.04058971815950488}


In [7]:
# Call tuiml_search to find components by keyword
result = execute_tool(
    "tuiml_search",
    query="tree",
    category="algorithm"
)

print(f"Found {result['count']} results for 'tree':")
for r in result["results"][:5]:
    print(f"  - {r['name']}: {r['description'][:60]}...")

Found 9 results for 'tree':
  - tuiml_algorithm_DecisionStumpClassifier: Decision Stump - a one-level decision tree....
  - tuiml_algorithm_C45TreeClassifier: C4.5 Decision Tree classifier....
  - tuiml_algorithm_RandomTreeClassifier: Random Tree classifier - a single randomized decision tree....
  - tuiml_algorithm_RandomForestClassifier: Random Forest classifier - ensemble of random trees....
  - tuiml_algorithm_ReducedErrorPruningTreeClassifier: Reduced Error Pruning Tree classifier....


---

## 5. Simulated Agent Workflow

Below we simulate an LLM agent that receives a task and autonomously works through it using TuiML tools. The comments represent the agent's "reasoning" between tool calls.

**Task:** *"Find the best classifier for the iris dataset."*

No actual LLM API is called here -- we execute the exact same tool calls that a real agent would make, in the order an agent would reason through them.

In [8]:
from tuiml.llm.tools import execute_tool
import json

print("="*60)
print("AGENT TASK: Find the best classifier for the iris dataset")
print("="*60)
print()

# ---------------------------------------------------------------
# Agent reasoning: "I need to discover what classifiers are
# available in TuiML. Let me call tuiml_list."
# ---------------------------------------------------------------

print("--- Step 1: Discover available classifiers ---")
print("Agent thinks: I should first see what classifiers exist.")
print()

step1 = execute_tool(
    "tuiml_list",
    category="algorithm",
    type="classifier",
    limit=50
)

classifier_names = [c["name"] for c in step1["components"]]
print(f"Tool call: tuiml_list(category='algorithm', type='classifier')")
print(f"Result: Found {step1['total']} classifiers.")
print(f"Examples: {classifier_names[:8]}")
print()

AGENT TASK: Find the best classifier for the iris dataset

--- Step 1: Discover available classifiers ---
Agent thinks: I should first see what classifiers exist.

Tool call: tuiml_list(category='algorithm', type='classifier')
Result: Found 70 classifiers.
Examples: ['tuiml_algorithm_NaiveBayesClassifier', 'tuiml_algorithm_NaiveBayesMultinomialClassifier', 'tuiml_algorithm_BayesianNetworkClassifier', 'tuiml_algorithm_DecisionStumpClassifier', 'tuiml_algorithm_C45TreeClassifier', 'tuiml_algorithm_RandomTreeClassifier', 'tuiml_algorithm_RandomForestClassifier', 'tuiml_algorithm_ReducedErrorPruningTreeClassifier']



In [9]:
# ---------------------------------------------------------------
# Agent reasoning: "I want to understand the RandomForest
# classifier before comparing. Let me get its parameter schema."
# ---------------------------------------------------------------

print("--- Step 2: Understand a classifier's parameters ---")
print("Agent thinks: Let me learn about RandomForestClassifier's hyperparameters.")
print()

step2 = execute_tool(
    "tuiml_describe",
    name="RandomForestClassifier"
)

print(f"Tool call: tuiml_describe(name='RandomForestClassifier')")
print(f"Result:")
print(f"  Type: {step2.get('type')}")
print(f"  Description: {step2.get('description', '')[:100]}...")
print(f"  Parameters: {json.dumps(step2.get('parameters', {}), indent=4)[:400]}...")
print()

--- Step 2: Understand a classifier's parameters ---
Agent thinks: Let me learn about RandomForestClassifier's hyperparameters.

Tool call: tuiml_describe(name='RandomForestClassifier')
Result:
  Type: classifier
  Description: Random Forest classifier - ensemble of random trees....
  Parameters: {
    "n_estimators": {
        "type": "integer",
        "default": 100,
        "minimum": 1,
        "description": "Number of trees in the forest"
    },
    "max_features": {
        "type": [
            "string",
            "integer",
            "number"
        ],
        "default": "sqrt",
        "description": "Number of features to consider at each split"
    },
    "max_depth": {
 ...



In [10]:
# ---------------------------------------------------------------
# Agent reasoning: "Now I will compare 5 diverse classifiers on
# the iris dataset using cross-validation. This will tell me
# which one generalizes best."
# ---------------------------------------------------------------

print("--- Step 3: Compare 5 classifiers with cross-validation ---")
print("Agent thinks: I will pick 5 diverse classifiers and run an experiment.")
print()

candidates = [
    "RandomForestClassifier",
    "NaiveBayesClassifier",
    "C45TreeClassifier",
    "KNearestNeighborsClassifier",
    "SVC",
]

step3 = execute_tool(
    "tuiml_experiment",
    algorithms=candidates,
    data="iris",
    target="class",
    cv=10
)

print(f"Tool call: tuiml_experiment(algorithms={candidates}, data='iris', target='class', cv=10)")
print(f"Status: {step3['status']}")
print()

# Parse results to find the best classifier
best_algo = None
best_score = -1.0

if step3["status"] == "success" and "results" in step3:
    print("Results (accuracy_score, 10-fold CV):")
    for dataset_name, models in step3["results"].items():
        for model_name, metrics in models.items():
            for metric_name, metric_data in metrics.items():
                mean_score = metric_data.get("mean", 0)
                std_score = metric_data.get("std", 0)
                print(f"  {model_name}: {metric_name} = {mean_score:.4f} +/- {std_score:.4f}")
                if mean_score > best_score:
                    best_score = mean_score
                    best_algo = model_name

print()
print(f"Agent concludes: The best classifier is {best_algo} with score {best_score:.4f}")

--- Step 3: Compare 5 classifiers with cross-validation ---
Agent thinks: I will pick 5 diverse classifiers and run an experiment.

Tool call: tuiml_experiment(algorithms=['RandomForestClassifier', 'NaiveBayesClassifier', 'C45TreeClassifier', 'KNearestNeighborsClassifier', 'SVC'], data='iris', target='class', cv=10)
Status: success

Results (accuracy_score, 10-fold CV):
  RandomForestClassifier: accuracy_score = 0.9400 +/- 0.0629
  NaiveBayesClassifier: accuracy_score = 0.9467 +/- 0.0653
  C45TreeClassifier: accuracy_score = 0.9333 +/- 0.0596
  KNearestNeighborsClassifier: accuracy_score = 0.9600 +/- 0.0442
  SVC: accuracy_score = 0.9733 +/- 0.0442

Agent concludes: The best classifier is SVC with score 0.9733


In [11]:
# ---------------------------------------------------------------
# Agent reasoning: "Now I will train the winning classifier on
# the full dataset to get a production-ready model."
# ---------------------------------------------------------------

print(f"--- Step 4: Train the winner ({best_algo}) ---")
print(f"Agent thinks: I should train {best_algo} and get a model I can save.")
print()

step4 = execute_tool(
    "tuiml_train",
    algorithm=best_algo,
    data="iris",
    target="class",
    cv=10
)

print(f"Tool call: tuiml_train(algorithm='{best_algo}', data='iris', target='class', cv=10)")
print(f"Status: {step4['status']}")
print(f"Model ID: {step4.get('model_id')}")
print(f"Model class: {step4.get('model_class')}")
print(f"Metrics: {step4.get('metrics')}")
print()

winner_model_id = step4.get("model_id")

--- Step 4: Train the winner (SVC) ---
Agent thinks: I should train SVC and get a model I can save.

Tool call: tuiml_train(algorithm='SVC', data='iris', target='class', cv=10)
Status: success
Model ID: e57d89fa39cf
Model class: SVC
Metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.044221663871405324, 'cv_f1_score_mean': 0.9392185592185592, 'cv_f1_score_std': 0.06902486969897233}



In [12]:
# ---------------------------------------------------------------
# Agent reasoning: "Finally, I should save this model so the
# user can load it later or deploy it."
# ---------------------------------------------------------------

print("--- Step 5: Save the trained model ---")
print("Agent thinks: I should persist the model to a user-friendly path.")
print()

if winner_model_id:
    step5 = execute_tool(
        "tuiml_save_model",
        model_id=winner_model_id,
        destination="./best_iris_classifier.joblib"
    )

    print(f"Tool call: tuiml_save_model(model_id='{winner_model_id}', destination='./best_iris_classifier.joblib')")
    print(f"Status: {step5['status']}")
    print(f"Message: {step5.get('message')}")
else:
    print("No model ID available (training may have failed).")

print()
print("="*60)
print("AGENT SUMMARY")
print("="*60)
print(f"Task: Find the best classifier for the iris dataset.")
print(f"Approach: Compared {len(candidates)} classifiers using 10-fold CV.")
print(f"Winner: {best_algo} (accuracy = {best_score:.4f})")
print(f"Model saved to: ./best_iris_classifier.joblib")

--- Step 5: Save the trained model ---
Agent thinks: I should persist the model to a user-friendly path.

Tool call: tuiml_save_model(model_id='e57d89fa39cf', destination='./best_iris_classifier.joblib')
Status: success
Message: Model saved to /Users/nileshverma/Documents/GitHub/tuiml/tuiml/tutorials/llm_friendly/best_iris_classifier.joblib

AGENT SUMMARY
Task: Find the best classifier for the iris dataset.
Approach: Compared 5 classifiers using 10-fold CV.
Winner: SVC (accuracy = 0.9733)
Model saved to: ./best_iris_classifier.joblib


---

## 6. Config-Driven Workflows with `tuiml.run()`

An LLM can generate a configuration dictionary and pass it to `tuiml.run()`, which executes the entire workflow in one call. This is useful when the agent has already decided on the full plan and wants to execute it atomically.

In [13]:
import tuiml

# An LLM might generate this config in one shot:
config = {
    "algorithm": "RandomForestClassifier",
    "data": "iris",
    "target": "class",
    "preprocessing": ["SimpleImputer", "MinMaxScaler"],
    "cv": 5,
    "n_estimators": 50,
    "random_state": 42,
}

print("LLM-generated config:")
print(json.dumps(config, indent=2))
print()

# Execute the entire workflow
result = tuiml.run(config)

print(f"Model: {result.model.__class__.__name__}")
print(f"Metrics: {result.metrics}")

LLM-generated config:
{
  "algorithm": "RandomForestClassifier",
  "data": "iris",
  "target": "class",
  "preprocessing": [
    "SimpleImputer",
    "MinMaxScaler"
  ],
  "cv": 5,
  "n_estimators": 50,
  "random_state": 42
}

Model: RandomForestClassifier
Metrics: {'cv_accuracy_score_mean': 0.9533333333333335, 'cv_accuracy_score_std': 0.02666666666666666, 'cv_f1_score_mean': 0.9315873015873016, 'cv_f1_score_std': 0.040388638581663285}


In [14]:
# Using a preset for preprocessing
config_with_preset = {
    "algorithm": "NaiveBayesClassifier",
    "data": "iris",
    "target": "class",
    "preset": "standard",
    "cv": 10,
}

print("Config with preset:")
print(json.dumps(config_with_preset, indent=2))
print()

result = tuiml.run(config_with_preset)
print(f"Model: {result.model.__class__.__name__}")
print(f"Metrics: {result.metrics}")

Config with preset:
{
  "algorithm": "NaiveBayesClassifier",
  "data": "iris",
  "target": "class",
  "preset": "standard",
  "cv": 10
}

Model: NaiveBayesClassifier
Metrics: {'cv_accuracy_score_mean': 0.35333333333333333, 'cv_accuracy_score_std': 0.13012814197295425, 'cv_f1_score_mean': 0.07857142857142857, 'cv_f1_score_std': 0.16428571428571428}


In [15]:
# Advanced: dict-style algorithm with feature selection
advanced_config = {
    "algorithm": {"name": "RandomForestClassifier", "n_estimators": 100, "max_depth": 10},
    "data": "iris",
    "target": "class",
    "preprocessing": [
        {"name": "SimpleImputer", "strategy": "median"},
        {"name": "MinMaxScaler"}
    ],
    "feature_selection": {"name": "SelectKBestSelector", "k": 3},
    "cv": 5,
}

print("Advanced LLM-generated config:")
print(json.dumps(advanced_config, indent=2))
print()

result = tuiml.run(advanced_config)
print(f"Model: {result.model.__class__.__name__}")
print(f"Metrics: {result.metrics}")

Advanced LLM-generated config:
{
  "algorithm": {
    "name": "RandomForestClassifier",
    "n_estimators": 100,
    "max_depth": 10
  },
  "data": "iris",
  "target": "class",
  "preprocessing": [
    {
      "name": "SimpleImputer",
      "strategy": "median"
    },
    {
      "name": "MinMaxScaler"
    }
  ],
  "feature_selection": {
    "name": "SelectKBestSelector",
    "k": 3
  },
  "cv": 5
}

Model: RandomForestClassifier
Metrics: {'cv_accuracy_score_mean': 0.9600000000000002, 'cv_accuracy_score_std': 0.024944382578492935, 'cv_f1_score_mean': 0.9395583160800551, 'cv_f1_score_std': 0.04058971815950488}


---

## 7. API Schema Discovery with `get_api_info()`

`get_api_info()` returns the full parameter schema for every public TuiML API function. An LLM can call this once at the start of a session to learn all available operations.

In [16]:
from tuiml.api import get_api_info

# Get all API function schemas
all_api = get_api_info()

print(f"TuiML API functions: {len(all_api)}")
print()
for name, info in all_api.items():
    print(f"  tuiml.{name}()")
    print(f"    {info['description']}")
    print(f"    Returns: {info['returns']}")
    print()

TuiML API functions: 14

  tuiml.train()
    Train a machine learning model with a complete workflow
    Returns: WorkflowResult with model, metrics, predictions

  tuiml.run()
    Execute workflow from configuration dict or JSON file
    Returns: WorkflowResult

  tuiml.pipeline()
    Create a fluent workflow for method chaining
    Returns: Workflow object for chaining

  tuiml.predict()
    Make predictions with a trained model
    Returns: numpy array of predictions

  tuiml.evaluate()
    Evaluate model performance on test data
    Returns: Dict of metric names to values

  tuiml.experiment()
    Compare multiple algorithms on multiple datasets
    Returns: Experiment object with results and statistics

  tuiml.list_algorithms()
    List available algorithms in registry
    Returns: List of algorithm metadata dicts

  tuiml.describe_algorithm()
    Get detailed info and parameter schema for an algorithm
    Returns: Dict with description, parameters schema, type

  tuiml.search_al

In [17]:
# Get schema for a specific function
train_info = get_api_info("train")

print("tuiml.train() parameter schema:")
print(json.dumps(train_info, indent=2))

tuiml.train() parameter schema:
{
  "description": "Train a machine learning model with a complete workflow",
  "parameters": {
    "algorithm": {
      "type": [
        "string",
        "object"
      ],
      "required": true,
      "description": "Algorithm name or dict with name and params"
    },
    "data": {
      "type": [
        "string",
        "object",
        "array"
      ],
      "required": true,
      "description": "File path, DataFrame, or numpy array"
    },
    "target": {
      "type": [
        "string",
        "array"
      ],
      "required": true,
      "description": "Target column name or array"
    },
    "preprocessing": {
      "type": [
        "string",
        "array",
        "null"
      ],
      "default": null,
      "description": "Preprocessing steps or preset name"
    },
    "feature_selection": {
      "type": [
        "string",
        "object",
        "null"
      ],
      "default": null,
      "description": "Feature selector name 

In [18]:
# Get schema for the experiment function
experiment_info = get_api_info("experiment")

print("tuiml.experiment() parameter schema:")
print(json.dumps(experiment_info, indent=2))

tuiml.experiment() parameter schema:
{
  "description": "Compare multiple algorithms on multiple datasets",
  "parameters": {
    "algorithms": {
      "type": [
        "object",
        "array"
      ],
      "required": true,
      "description": "Dict or list of algorithms to compare"
    },
    "datasets": {
      "type": [
        "object",
        "array"
      ],
      "required": true,
      "description": "Dict or list of datasets"
    },
    "cv": {
      "type": "integer",
      "default": 10,
      "description": "Number of CV folds"
    },
    "metrics": {
      "type": [
        "array",
        "null"
      ],
      "default": null,
      "description": "Metrics to compute"
    }
  },
  "returns": "Experiment object with results and statistics"
}


---

## 8. Integration Patterns

TuiML tools can be plugged into any LLM provider's function calling API. Below we show the exact format conversion for **Anthropic Claude** and **OpenAI**, plus the tool execution loop.

### 8.1 Anthropic Claude (Tool Use)

Anthropic expects tools in this format:
```json
{
    "name": "tool_name",
    "description": "...",
    "input_schema": { ... }
}
```

In [19]:
from tuiml.llm.tools import DISCOVERY_TOOLS, WORKFLOW_TOOLS

def to_anthropic_tools(tools_dict: dict) -> list:
    """Convert TuiML tool schemas to Anthropic Claude format."""
    return [
        {
            "name": name,
            "description": schema["description"],
            "input_schema": schema["inputSchema"],
        }
        for name, schema in tools_dict.items()
    ]

all_tools = {**DISCOVERY_TOOLS, **WORKFLOW_TOOLS}
anthropic_tools = to_anthropic_tools(all_tools)

print(f"Converted {len(anthropic_tools)} tools to Anthropic format.")
print()
print("Sample (tuiml_train):")
sample = next(t for t in anthropic_tools if t["name"] == "tuiml_train")
print(json.dumps(sample, indent=2)[:500] + "...")

Converted 12 tools to Anthropic format.

Sample (tuiml_train):
{
  "name": "tuiml_train",
  "description": "Train a machine learning model. Supports supervised (classifiers, regressors) and unsupervised (clusterers) learning. Returns trained model and metrics.",
  "input_schema": {
    "type": "object",
    "properties": {
      "algorithm": {
        "type": "string",
        "description": "Algorithm class name. Examples:\n- Classifiers: 'RandomForestClassifier', 'SVM', 'NaiveBayesClassifier', 'C45TreeClassifier'\n- Regressors: 'LinearRegressio...


In [20]:
# Pseudocode for the full Anthropic tool-use loop:
#
# import anthropic
# client = anthropic.Anthropic()
#
# response = client.messages.create(
#     model="claude-sonnet-4-20250514",
#     tools=anthropic_tools,
#     messages=[{"role": "user", "content": "Train RF on iris"}]
# )
#
# while response.stop_reason == "tool_use":
#     for block in response.content:
#         if block.type == "tool_use":
#             # Execute TuiML tool
#             result = execute_tool(block.name, **block.input)
#             # Send result back to Claude
#             ...

print("Anthropic tool-use loop (pseudocode above)")

Anthropic tool-use loop (pseudocode above)


### 8.2 OpenAI Function Calling

OpenAI wraps each tool in a `{"type": "function", "function": {...}}` container and uses `parameters` instead of `input_schema`.

In [21]:
def to_openai_tools(tools_dict: dict) -> list:
    """Convert TuiML tool schemas to OpenAI function calling format."""
    return [
        {
            "type": "function",
            "function": {
                "name": name,
                "description": schema["description"],
                "parameters": schema["inputSchema"],
            },
        }
        for name, schema in tools_dict.items()
    ]

openai_tools = to_openai_tools(all_tools)

print(f"Converted {len(openai_tools)} tools to OpenAI format.")
print()
print("Sample (tuiml_train):")
sample = next(t for t in openai_tools if t["function"]["name"] == "tuiml_train")
print(json.dumps(sample, indent=2)[:500] + "...")

Converted 12 tools to OpenAI format.

Sample (tuiml_train):
{
  "type": "function",
  "function": {
    "name": "tuiml_train",
    "description": "Train a machine learning model. Supports supervised (classifiers, regressors) and unsupervised (clusterers) learning. Returns trained model and metrics.",
    "parameters": {
      "type": "object",
      "properties": {
        "algorithm": {
          "type": "string",
          "description": "Algorithm class name. Examples:\n- Classifiers: 'RandomForestClassifier', 'SVM', 'NaiveBayesClassifier', 'C45Dec...


In [22]:
# Pseudocode for the full OpenAI function calling loop:
#
# from openai import OpenAI
# client = OpenAI()
#
# response = client.chat.completions.create(
#     model="gpt-4o",
#     tools=openai_tools,
#     messages=[{"role": "user", "content": "Train RF on iris"}]
# )
#
# while response.choices[0].message.tool_calls:
#     for call in response.choices[0].message.tool_calls:
#         args = json.loads(call.function.arguments)
#         result = execute_tool(call.function.name, **args)
#         # Send result back to ChatGPT
#         ...

print("OpenAI function calling loop (pseudocode above)")

OpenAI function calling loop (pseudocode above)


### 8.3 Format Comparison

| Aspect | Anthropic Claude | OpenAI ChatGPT |
|--------|-----------------|----------------|
| Schema key | `input_schema` | `parameters` |
| Tool wrapper | Flat dict | `{"type": "function", "function": {...}}` |
| Tool call signal | `stop_reason == "tool_use"` | `message.tool_calls` is truthy |
| Result format | `tool_result` content block | `{"role": "tool", "content": ...}` message |
| Execution | `execute_tool(name, **input)` | `execute_tool(name, **json.loads(args))` |

---

## 9. MCP Integration

For the most seamless experience with Claude, TuiML includes a full **MCP (Model Context Protocol) server**. Instead of manually converting tool formats, the MCP server exposes all tools natively and handles the protocol.

### Quick Start

```bash
# Run the MCP server
tuiml-mcp

# Or via Python module
python -m tuiml.llm.server
```

### Claude Desktop Configuration

Add to `claude_desktop_config.json`:

```json
{
    "mcpServers": {
        "tuiml": {
            "command": "tuiml-mcp"
        }
    }
}
```

Once configured, Claude Desktop can directly call all 12 TuiML tools without any additional code.

In [23]:
from tuiml.llm import MCP_AVAILABLE, get_tools_for_llm

print(f"MCP package available: {MCP_AVAILABLE}")

# get_tools_for_llm() returns the same 12 tools that the MCP server exposes
mcp_tools = get_tools_for_llm()
print(f"MCP tools exposed: {len(mcp_tools)}")
for t in mcp_tools:
    print(f"  - {t['name']}")

MCP package available: True
MCP tools exposed: 12
  - tuiml_train
  - tuiml_predict
  - tuiml_evaluate
  - tuiml_experiment
  - tuiml_upload_data
  - tuiml_save_model
  - tuiml_serve_model
  - tuiml_stop_server
  - tuiml_server_status
  - tuiml_list
  - tuiml_describe
  - tuiml_search
